Populando

In [259]:
import pandas as pd
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_validate


In [260]:
OUTPUT_DIR = Path('../data')

cardinali = pd.read_csv(OUTPUT_DIR / 'cardinali.csv')
roca = pd.read_csv(OUTPUT_DIR / 'roca.csv')
center = pd.read_csv(OUTPUT_DIR / 'center.csv')
abias = pd.read_csv(OUTPUT_DIR / 'abias.csv')
sape = pd.read_csv(OUTPUT_DIR / 'sape.csv')

df = pd.concat([cardinali, roca, center, abias, sape], ignore_index=True)

print(f'cardinali: {len(cardinali)} linhas')
print(f'roca:      {len(roca)} linhas')
print(f'center:    {len(center)} linhas')
print(f'abias:     {len(abias)} linhas')
print(f'sape:      {len(sape)} linhas')
print(f'total:     {len(df)} linhas')
df.head(1)

cardinali: 4232 linhas
roca:      2438 linhas
center:    678 linhas
abias:     274 linhas
sape:      1035 linhas
total:     8657 linhas


,fonte,codigo,titulo,tipo,subtipo,finalidade,preco_locacao,preco_venda,valor_condominio,valor_iptu,...,banheiros,garagens,area_total,area_construida,area_util,area_terreno,descricao,url,latitude,longitude
0,Cardinali,237897,Valor do aluguel está incluso condômino e IPTU,Apartamento,Padrao,Locacao,"1.112,00",NaN,",",.,...,1.0,1.0,NaN,NaN,50.0,NaN,Valor do aluguel está incluso condômino e IPTU...,https://www.cardinali.com.br/alugar/Sao-Carlos...,-22.026474,-47.917273


Pre processamento

In [261]:
df.columns = df.columns.str.lower().str.strip()

colunas_texto = ["tipo", "subtipo", "finalidade", "bairro", "cidade", "estado", "fonte"]
for coluna in colunas_texto:
    if coluna in df.columns:
        df[coluna] = df[coluna].astype(str).str.lower().str.strip()
        df[coluna] = df[coluna].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')

subs = {r'\bjd\b': 'jardim', r'\bj\b': 'jardim', r'\bpq\b': 'parque',
        r'\bvl\b': 'vila', r'\bres\b': 'residencial', r'\bcond\b': 'condominio', r'\bch\b': 'chacara'}
df['bairro'] = df['bairro'].replace(subs, regex=True)
df['bairro'] = df['bairro'].str.replace(r'\s+', ' ', regex=True).str.strip()

if 'finalidade' in df.columns:
    df = df[df["finalidade"] == "locacao"]

if 'tipo' in df.columns:
    df['tipo'] = df['tipo'].replace({'apartamentos': 'apartamento', 'casas': 'casa'})
    df = df[df['tipo'].isin(['apartamento', 'casa'])]
    
# # Excluir imóveis da Cardinali
# if 'fonte' in df.columns:
#     df = df[df['fonte'] != 'cardinali']
#     # Remover a coluna fonte
#     df = df.drop(columns=['fonte'])

print(f"Base Pronta! Total de linhas para o modelo: {len(df)}")

Base Pronta! Total de linhas para o modelo: 2000


In [262]:
def parse_br_money(s):
    s = str(s)
    if s in ['nan', 'None', '', '.', ',']: return np.nan
    s = s.replace('.', '').replace(',', '.')
    return pd.to_numeric(s, errors='coerce')

for col in ['preco_locacao', 'valor_condominio', 'valor_iptu']:
    if col in df.columns: df[col] = df[col].apply(parse_br_money)

for col in ['area_util', 'area_construida', 'area_total']:
    if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce')

for col in ['dormitorios', 'banheiros', 'garagens']:
    if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

if 'latitude' in df.columns: df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
if 'longitude' in df.columns: df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

if 'area_util' in df.columns:
    df['area_util'] = df['area_util'].fillna(df.get('area_construida')).fillna(df.get('area_total'))

df = df.dropna(subset=['preco_locacao', 'bairro', 'area_util'])
df = df[(df['preco_locacao'] > 0) & (df['area_util'] > 5)]
df = df[~((df['tipo'] == 'apartamento') & (df['area_util'] > 500))]
df = df[~((df['tipo'] == 'casa') & (df['area_util'] > 2000))]

# Imputação Condomínio
if 'valor_condominio' in df.columns:
    df['valor_condominio'] = df['valor_condominio'].replace(0, np.nan)
    mask_apt = (df['tipo'] == 'apartamento')
    df.loc[mask_apt, 'valor_condominio'] = df.loc[mask_apt, 'valor_condominio'].fillna(
        df[mask_apt].groupby('bairro')['valor_condominio'].transform('median')
    )
    df.loc[mask_apt, 'valor_condominio'] = df.loc[mask_apt, 'valor_condominio'].fillna(df.loc[mask_apt, 'valor_condominio'].median())
    df['valor_condominio'] = df['valor_condominio'].fillna(0)

# Imputação IPTU
if 'valor_iptu' in df.columns:
    df['valor_iptu'] = df['valor_iptu'].replace(0, np.nan)
    df['valor_iptu'] = df['valor_iptu'].fillna(df.groupby('bairro')['valor_iptu'].transform('median')).fillna(0)

print(f"Base Pronta! Total de linhas para o modelo: {len(df)}")

Base Pronta! Total de linhas para o modelo: 1446


In [263]:
desc = df['descricao'].fillna('').astype(str).str.lower() if 'descricao' in df.columns else pd.Series('', index=df.index)

# Features limpas e diretas
df['foco_estudante'] = desc.str.contains('estudante|republica|caaso|federal|kitnet|republicas', regex=True).astype(int)
df['reformado_novo'] = desc.str.contains('reformado|novo|primeira locacao|recem', regex=True).astype(int)

df['cond_por_m2'] = (df['valor_condominio'] / df['area_util']).replace([np.inf, -np.inf], 0).fillna(0)
df['total_comodos'] = df['dormitorios'] + df['banheiros']
df['area_por_quarto'] = (df['area_util'] / df['dormitorios'].clip(lower=1))

if 'latitude' in df.columns and 'longitude' in df.columns:
    KM_DEGREE = 111 
    CENTRO_LAT, CENTRO_LON = -22.0174, -47.8908
    UFSCAR_LAT, UFSCAR_LON = -21.9839, -47.8822
    USP_CAMPUS1_LAT, USP_CAMPUS1_LON = -22.0063, -47.8946

    # Mantemos só os 3 polos principais para não dar colinearidade
    df['dist_centro'] = np.sqrt((df['latitude'] - CENTRO_LAT)**2 + (df['longitude'] - CENTRO_LON)**2) * KM_DEGREE
    df['dist_ufscar'] = np.sqrt((df['latitude'] - UFSCAR_LAT)**2 + (df['longitude'] - UFSCAR_LON)**2) * KM_DEGREE
    df['dist_usp'] = np.sqrt((df['latitude'] - USP_CAMPUS1_LAT)**2 + (df['longitude'] - USP_CAMPUS1_LON)**2) * KM_DEGREE

    colunas_dist = ['dist_centro', 'dist_ufscar', 'dist_usp']
    df[colunas_dist] = df[colunas_dist].fillna(df[colunas_dist].median())

contagem = df['bairro'].value_counts()
df.loc[~df['bairro'].isin(contagem[contagem >= 5].index), 'bairro'] = 'outros'
df = df.drop_duplicates(subset=['preco_locacao', 'bairro', 'area_util', 'dormitorios', 'banheiros', 'tipo'])

print(f"Base Pronta! Total de linhas para o modelo: {len(df)}")

Base Pronta! Total de linhas para o modelo: 1341


In [264]:
# O filtro robusto usando o pacote total
df['preco_total_temp'] = df['preco_locacao'] + df['valor_condominio'] + df['valor_iptu']
df['preco_m2_temp'] = df['preco_total_temp'] / df['area_util']
Q_low, Q_high = df['preco_m2_temp'].quantile(0.02), df['preco_m2_temp'].quantile(0.98)

df = df[(df['preco_m2_temp'] >= Q_low) & (df['preco_m2_temp'] <= Q_high)]

# Limpa colunas temporárias
df = df.drop(columns=['preco_total_temp', 'preco_m2_temp'])

print(f"Base Pronta! Total de linhas para o modelo: {len(df)}")

Base Pronta! Total de linhas para o modelo: 1287


In [265]:
df.head(5)

,fonte,codigo,titulo,tipo,subtipo,finalidade,preco_locacao,preco_venda,valor_condominio,valor_iptu,...,latitude,longitude,foco_estudante,reformado_novo,cond_por_m2,total_comodos,area_por_quarto,dist_centro,dist_ufscar,dist_usp
0,cardinali,237897,Valor do aluguel está incluso condômino e IPTU,apartamento,padrao,locacao,1112.0,NaN,1100.00,215.00,...,-22.026474,-47.917273,0,0,22.000000,3,25.0,3.106273,6.122733,3.368664
4,cardinali,237880,Apartamento de 2 dormitórios.,apartamento,padrao,locacao,1889.0,NaN,4100.00,87.50,...,-22.030589,-47.887940,0,0,82.000000,3,25.0,1.497940,5.221448,2.795529
7,cardinali,237869,Próximo USP,apartamento,padrao,locacao,1667.0,NaN,295.57,25.93,...,-22.009899,-47.902220,0,0,9.852333,2,30.0,1.516553,3.642336,0.935390
9,cardinali,237867,Edícula com 2 Dormitórios,casa,edicula,locacao,1334.0,NaN,0.00,40.00,...,-21.995478,-47.889818,0,0,0.000000,3,20.0,2.435794,1.538376,1.313310
13,cardinali,237842,Próximo USP,apartamento,padrao,locacao,1778.0,NaN,344.59,90.00,...,-22.000347,-47.902719,0,0,7.178958,3,24.0,2.309428,2.918940,1.117513


In [266]:
features_limpas = [
    'area_util', 'bairro', 'area_por_quarto', 'dist_centro', 
    'dist_ufscar', 'dist_usp', 'garagens', 
    'total_comodos', 'dormitorios', 'banheiros', 'reformado_novo', 
    'valor_condominio', 'valor_iptu', 'cond_por_m2', 'foco_estudante'
]

X = df[features_limpas].copy()
y = df['preco_locacao'].copy() 

preprocessor = ColumnTransformer(
    transformers=[('bairro_enc', TargetEncoder(random_state=42), ['bairro'])],
    remainder='passthrough'
)

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', TransformedTargetRegressor(
        regressor=LGBMRegressor(
            n_estimators=300, 
            learning_rate=0.05, 
            max_depth=7, 
            num_leaves=31, 
            random_state=42, 
            n_jobs=-1,
            objective='mae',
            verbose=-1
        ),
        func=np.log1p, inverse_func=np.expm1
    ))
])

kf = KFold(n_splits=20, shuffle=True, random_state=42)
res = cross_validate(pipe, X, y, cv=kf, scoring={'mae': 'neg_mean_absolute_error', 'r2': 'r2'})

print("="*60)
print(f"ALVO: ALUGUEL PURO (preco_locacao)")
print("-" * 60)
print(f"MAE Médio: R$ {-res['test_mae'].mean():.2f}")
print(f"MAE Folds: {-np.round(res['test_mae'], 2)}")
print("-" * 60)
print(f"R² Médio:  {res['test_r2'].mean():.3f}")
print(f"R² Folds:  {np.round(res['test_r2'], 3)}")
print("="*60)

ALVO: ALUGUEL PURO (preco_locacao)
------------------------------------------------------------
MAE Médio: R$ 604.86
MAE Folds: [ 516.33  685.69  571.24 1167.27  571.1   487.48  484.32  653.82  821.79
  506.36  372.54  754.29  596.28  488.21  733.64  569.92  523.61  596.91
  584.46  411.86]
------------------------------------------------------------
R² Médio:  0.649
R² Folds:  [0.586 0.728 0.714 0.317 0.574 0.585 0.733 0.689 0.281 0.753 0.892 0.586
 0.571 0.74  0.602 0.723 0.674 0.729 0.687 0.817]


In [267]:
pipe.fit(X, y)
feat_names = ['bairro_encoded'] + [c for c in X.columns if c != 'bairro']
importances = pipe.named_steps['regressor'].regressor_.feature_importances_
importances = importances / importances.sum()

print("\n=== Peso das Features no Preço (LightGBM) ===")
for n, i in sorted(zip(feat_names, importances), key=lambda x: -x[1]):
    if i > 0.001:
        bar = '█' * int(i * 100)
        print(f"  {n:22s}: {i:.4f} {bar}")


=== Peso das Features no Preço (LightGBM) ===
  valor_iptu            : 0.1391 █████████████
  bairro_encoded        : 0.1338 █████████████
  area_por_quarto       : 0.1170 ███████████
  area_util             : 0.0992 █████████
  dist_centro           : 0.0948 █████████
  dist_ufscar           : 0.0892 ████████
  dist_usp              : 0.0877 ████████
  valor_condominio      : 0.0845 ████████
  cond_por_m2           : 0.0689 ██████
  garagens              : 0.0300 ██
  banheiros             : 0.0211 ██
  total_comodos         : 0.0175 █
  dormitorios           : 0.0083 
  reformado_novo        : 0.0050 
  foco_estudante        : 0.0040 


In [268]:
import joblib
from pathlib import Path

model_dir = Path('../models')
model_dir.mkdir(exist_ok=True)
joblib.dump(pipe, model_dir / 'modelo_aluguel.pkl')
print("Modelo salvo em ../models/modelo_aluguel.pkl")

Modelo salvo em ../models/modelo_aluguel.pkl


In [269]:
KM_DEGREE = 111
CENTRO_LAT, CENTRO_LON = -22.0174, -47.8908
UFSCAR_LAT, UFSCAR_LON = -21.9839, -47.8822
USP_CAMPUS1_LAT, USP_CAMPUS1_LON = -22.0063, -47.8946

EMBARE_LAT, EMBARE_LON = -21.9960, -47.8560

imoveis = pd.DataFrame([
    {
        'nome': 'Jardim Embaré - 2q, 1ban, 54m²',
        'area_util': 54,
        'bairro': 'jardim embare',
        'dormitorios': 2,
        'banheiros': 1,
        'garagens': 1,
        'valor_condominio': 200,
        'valor_iptu': 42,
        'reformado_novo': 0,
        'foco_estudante': 0,
        'total_comodos': 2 + 1,
        'area_por_quarto': 54 / 2,
        'cond_por_m2': 200 / 54,
        'dist_centro': np.sqrt((EMBARE_LAT - CENTRO_LAT)**2 + (EMBARE_LON - CENTRO_LON)**2) * KM_DEGREE,
        'dist_ufscar': np.sqrt((EMBARE_LAT - UFSCAR_LAT)**2 + (EMBARE_LON - UFSCAR_LON)**2) * KM_DEGREE,
        'dist_usp': np.sqrt((EMBARE_LAT - USP_CAMPUS1_LAT)**2 + (EMBARE_LON - USP_CAMPUS1_LON)**2) * KM_DEGREE,
    },
    {
        'nome': 'Centro - 2q, 1ban, 50m²',
        'area_util': 50,
        'bairro': 'centro',
        'dormitorios': 2,
        'banheiros': 1,
        'garagens': 0,
        'valor_condominio': 1000,
        'valor_iptu': 0,
        'reformado_novo': 1,
        'foco_estudante': 0,
        'total_comodos': 2 + 1,
        'area_por_quarto': 50 / 2,
        'cond_por_m2': 1000 / 50,
        'dist_centro': 0,
        'dist_ufscar': np.sqrt((CENTRO_LAT - UFSCAR_LAT)**2 + (CENTRO_LON - UFSCAR_LON)**2) * KM_DEGREE,
        'dist_usp': np.sqrt((CENTRO_LAT - USP_CAMPUS1_LAT)**2 + (CENTRO_LON - USP_CAMPUS1_LON)**2) * KM_DEGREE,
    },
])

precos = pipe.predict(imoveis[features_limpas])
for i, row in imoveis.iterrows():
    print(f"{row['nome']}: R$ {precos[i]:.2f}")

Jardim Embaré - 2q, 1ban, 54m²: R$ 1261.39
Centro - 2q, 1ban, 50m²: R$ 1241.17
